# Experiment 14: Case Study on Scene Understanding using RBMs

**Aim:** Study how a **Restricted Boltzmann Machine (RBM)** - a two-layer generative neural network - can learn useful *visual features* from raw scene images **without labels**, and how those features help recognise the scene content.

**Case-study pipeline**
1. Implement a Bernoulli-Bernoulli RBM from scratch (TensorFlow) trained with **Contrastive Divergence (CD-k)**.
2. Train it unsupervised on natural-scene images (CIFAR-10, converted to 32x32 greyscale = 1024 visible units).
3. Visualise what it learned: weight filters, reconstructions, and samples "dreamed" by Gibbs sampling.
4. Use the hidden-unit activations as features for a simple classifier and compare with raw pixels (supervised scene/object recognition).
5. Group classes into coarse scene types (*man-made / vehicles* vs *natural / animals*) for a scene-level analysis.

**RBM energy:** `E(v,h) = -a.v - b.h - v.W.h`, `P(h=1|v) = sigmoid(b + vW)`, `P(v=1|h) = sigmoid(a + Wh)`.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, time
import tensorflow as tf
from tensorflow import keras
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.decomposition import PCA

tf.random.set_seed(11); np.random.seed(11)
CLASSES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
N_HID, EPOCHS, BATCH, LR, CD_K = 256, 20, 100, 0.05, 1
N_TRAIN, N_TEST = 20000, 5000
print("TensorFlow", tf.__version__)

## 1. Data: greyscale scenes
Each colour image is converted to luminance and **contrast-normalised per image** (zero mean / unit variance) then squashed into [0,1] to be usable as Bernoulli probabilities by the RBM.

In [ ]:
(x_tr, y_tr), (x_te, y_te) = keras.datasets.cifar10.load_data()
y_tr, y_te = y_tr.ravel()[:N_TRAIN], y_te.ravel()[:N_TEST]

def prep(x):
    g = (0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]).astype("float32") / 255.0
    g = g.reshape(len(g), -1)
    g = (g - g.mean(1, keepdims=True)) / (g.std(1, keepdims=True) + 1e-6)
    return 1 / (1 + np.exp(-1.5 * g))                        # squash to (0,1)

V_tr, V_te = prep(x_tr[:N_TRAIN]), prep(x_te[:N_TEST])
print(V_tr.shape, V_te.shape)
fig, ax = plt.subplots(2, 8, figsize=(14, 3.6))
for i in range(8):
    ax[0, i].imshow(x_tr[i]); ax[0, i].axis("off"); ax[0, i].set_title(CLASSES[y_tr[i]], fontsize=8)
    ax[1, i].imshow(V_tr[i].reshape(32, 32), cmap="gray"); ax[1, i].axis("off")
plt.show()

## 2. The RBM
Training uses **CD-k**: positive phase `<v h>_data` minus negative phase `<v h>_model` obtained after k steps of Gibbs sampling, with momentum and weight decay.

In [ ]:
class RBM:
    def __init__(self, n_vis, n_hid, lr=0.05, momentum=0.5, weight_decay=1e-4):
        self.W = tf.Variable(tf.random.normal([n_vis, n_hid], stddev=0.01)); self.a = tf.Variable(tf.zeros([n_vis])); self.b = tf.Variable(tf.zeros([n_hid]))
        self.lr, self.mom, self.wd = lr, momentum, weight_decay
        self.vW, self.va, self.vb = [tf.zeros_like(p) for p in (self.W, self.a, self.b)]

    def p_h(self, v): return tf.sigmoid(tf.matmul(v, self.W) + self.b)
    def p_v(self, h): return tf.sigmoid(tf.matmul(h, self.W, transpose_b=True) + self.a)
    @staticmethod
    def sample(p): return tf.cast(tf.random.uniform(tf.shape(p)) < p, tf.float32)

    def gibbs(self, v, k=1):
        for _ in range(k):
            h = self.sample(self.p_h(v)); v = self.p_v(h)
        return v

    @tf.function
    def train_step(self, v0, k):
        ph0 = self.p_h(v0); h = self.sample(ph0)
        for _ in range(k):                                    # k-step Gibbs chain
            pv = self.p_v(h); ph = self.p_h(pv); h = self.sample(ph)
        n = tf.cast(tf.shape(v0)[0], tf.float32)
        dW = (tf.matmul(v0, ph0, transpose_a=True) - tf.matmul(pv, ph, transpose_a=True)) / n - self.wd * self.W
        da = tf.reduce_mean(v0 - pv, 0); db = tf.reduce_mean(ph0 - ph, 0)
        self.vW.assign(self.mom * self.vW + self.lr * dW); self.va.assign(self.mom * self.va + self.lr * da); self.vb.assign(self.mom * self.vb + self.lr * db)
        self.W.assign_add(self.vW); self.a.assign_add(self.va); self.b.assign_add(self.vb)
        return tf.reduce_mean(tf.square(v0 - pv))             # reconstruction error

    def free_energy(self, v):
        return -tf.matmul(v, self.a[:, None])[:, 0] - tf.reduce_sum(tf.math.softplus(tf.matmul(v, self.W) + self.b), 1)

rbm = RBM(1024, N_HID, LR)

## 3. Unsupervised training

In [ ]:
errs, fe_tr, fe_te = [], [], []
ds_np = V_tr.astype("float32"); t0 = time.time()
for ep in range(EPOCHS):
    if ep == 5: rbm.mom = 0.9                                   # momentum schedule (Hinton's practical guide)
    perm = np.random.permutation(len(ds_np)); e = []
    for i in range(0, len(perm) - BATCH + 1, BATCH):
        e.append(float(rbm.train_step(tf.constant(ds_np[perm[i:i+BATCH]]), CD_K)))
    errs.append(np.mean(e))
    fe_tr.append(float(tf.reduce_mean(rbm.free_energy(ds_np[:2000])))); fe_te.append(float(tf.reduce_mean(rbm.free_energy(V_te[:2000].astype("float32")))))
    print(f"epoch {ep+1:2d}  recon MSE={errs[-1]:.4f}  free-energy train={fe_tr[-1]:.1f} test={fe_te[-1]:.1f}")
print(f"training time {time.time()-t0:.0f}s")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(errs); ax[0].set_title("Reconstruction error (MSE)"); ax[0].set_xlabel("epoch")
ax[1].plot(fe_tr, label="train"); ax[1].plot(fe_te, label="test"); ax[1].set_title("Mean free energy (gap = over-fitting)"); ax[1].legend(); ax[1].set_xlabel("epoch")
plt.show()

## 4. What did the RBM learn?
### 4a. Hidden-unit receptive fields
Each hidden unit becomes a detector of an edge / blob / texture pattern - the building blocks of scenes.

In [ ]:
W = rbm.W.numpy()
def tile(imgs, rows, cols, size=32):
    canvas = np.ones((rows * (size + 1), cols * (size + 1)))
    for i in range(rows * cols):
        im = imgs[i].reshape(size, size); im = (im - im.min()) / (im.max() - im.min() + 1e-9)
        r, c = divmod(i, cols); canvas[r*(size+1):r*(size+1)+size, c*(size+1):c*(size+1)+size] = im
    return canvas
plt.figure(figsize=(9, 9)); plt.imshow(tile(W.T, 12, 12), cmap="gray"); plt.axis("off"); plt.title("144 learned RBM filters (weights of 144 hidden units)"); plt.show()

### 4b. Reconstructions (visible -> hidden -> visible)

In [ ]:
v = V_te[:12].astype("float32"); rec = rbm.gibbs(tf.constant(v), 1).numpy()
fig, ax = plt.subplots(2, 12, figsize=(16, 3.2))
for i in range(12):
    ax[0, i].imshow(v[i].reshape(32, 32), cmap="gray"); ax[0, i].axis("off"); ax[1, i].imshow(rec[i].reshape(32, 32), cmap="gray"); ax[1, i].axis("off")
ax[0, 0].set_title("original", fontsize=9, loc="left"); ax[1, 0].set_title("RBM reconstruction", fontsize=9, loc="left"); plt.show()

### 4c. Samples "dreamed" by the model (long Gibbs chains from noise)

In [ ]:
v = tf.constant(np.random.rand(16, 1024).astype("float32")); frames = []
for step in range(1000):
    v = rbm.gibbs(v, 1)
    if step in (0, 100, 1000 - 1): frames.append(v.numpy())
fig, ax = plt.subplots(3, 1, figsize=(14, 6.5))
for a, f, t in zip(ax, frames, ["after 1 step", "after 100 steps", "after 1000 steps"]):
    a.imshow(tile(f, 1, 16), cmap="gray"); a.axis("off"); a.set_title(t, fontsize=9)
plt.show()

## 5. Using RBM features for scene recognition
A logistic regression classifier is trained on (a) raw pixels, (b) PCA(256) features and (c) the RBM's 256 hidden probabilities, on the same data. The RBM features were learned **without labels**.

In [ ]:
H_tr = rbm.p_h(tf.constant(V_tr.astype("float32"))).numpy(); H_te = rbm.p_h(tf.constant(V_te.astype("float32"))).numpy()
pca = PCA(256, random_state=0).fit(V_tr); P_tr, P_te = pca.transform(V_tr), pca.transform(V_te)

def lr_eval(Xa, Xb, y_a, y_b, C=1.0):
    clf = LogisticRegression(max_iter=300, C=C, n_jobs=-1).fit(Xa, y_a); return clf, accuracy_score(y_b, clf.predict(Xb))

res = {}
for name, (A, B) in {"raw pixels (1024)": (V_tr, V_te), "PCA (256)": (P_tr, P_te), "RBM hidden (256)": (H_tr, H_te)}.items():
    clf, acc = lr_eval(A, B, y_tr, y_te, C=0.1 if name.startswith("raw") else 1.0); res[name] = (clf, acc); print(f"{name:20s} 10-class accuracy: {acc*100:.2f}%")
plt.figure(figsize=(6, 3)); plt.bar(res.keys(), [a*100 for _, a in res.values()], color=["#999", "#69c", "#e84"]); plt.ylabel("accuracy %"); plt.title("Logistic regression on different features"); plt.xticks(rotation=15); plt.show()

## 6. Coarse scene-level analysis
Classes are grouped into two scene types - **man-made / vehicle scenes** (airplane, automobile, ship, truck) vs **natural / animal scenes** (the other six).

In [ ]:
vehicle = np.isin(np.arange(10), [0, 1, 8, 9])
s_tr, s_te = vehicle[y_tr].astype(int), vehicle[y_te].astype(int)
for name, (A, B) in {"raw pixels": (V_tr, V_te), "RBM hidden": (H_tr, H_te)}.items():
    _, acc = lr_eval(A, B, s_tr, s_te); print(f"{name:12s} vehicle-vs-animal accuracy: {acc*100:.2f}%  (majority baseline {max(s_te.mean(), 1-s_te.mean())*100:.1f}%)")

clf = res["RBM hidden (256)"][0]
ConfusionMatrixDisplay(confusion_matrix(y_te, clf.predict(H_te)), display_labels=CLASSES).plot(cmap="Blues", colorbar=False, xticks_rotation=60, values_format="d")
plt.title("10-class confusion matrix (RBM features)"); plt.show()

## Conclusion / Case-study findings
* The RBM learned, with no labels, oriented edge, blob and texture detectors from scene images, and can reconstruct inputs and generate scene-like textures by Gibbs sampling.
* Its hidden representation is a compact, informative description of a scene: a linear classifier on it is competitive with / better than one on raw pixels and similar to PCA; the coarse "man-made vs natural" distinction is recognised much more reliably than fine-grained classes because global layout and contrast statistics are what a shallow RBM captures.
* Limitations: a single RBM is shallow and has no translation invariance; stacking RBMs into a **Deep Belief Network** (greedy layer-wise pretraining) or using convolutional RBMs gives richer scene representations - the historical stepping stone to modern deep networks.